In [1]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch

c:\Users\Maga\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load tokenized splits from preprocessing
train_tok = load_from_disk("train_tok")
val_tok   = load_from_disk("val_tok")
test_tok  = load_from_disk("test_tok")

print(f"Train: {len(train_tok)} | Val: {len(val_tok)} | Test: {len(test_tok)}")

Train: 13994 | Val: 2999 | Test: 3000


In [3]:
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3

LABEL2ID = {"positive": 0, "negative": 1, "neutral": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [4]:
# Evaluation metrics — macro F1 + accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

---
## Part 1 — Full Fine-Tuning

In [8]:
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

print(f"Total parameters: {sum(p.numel() for p in full_model.parameters()):,}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 10099.10it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 278,045,955


In [ ]:
full_ft_args = TrainingArguments(
    output_dir="./hf_full_ft_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    warmup_steps=200, # first 200 steps learning rate goes from 0 to 2e-5
    weight_decay=0.01, # regularization 
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    bf16=True,
    logging_steps=50, # eval at every 50 steps 
)

full_trainer = Trainer(
    model=full_model,
    args=full_ft_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

full_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.879126,0.873244,0.593198,0.595381
2,0.799978,0.833288,0.626209,0.631710
3,0.716758,0.825512,0.624875,0.627430
4,0.664425,0.896104,0.630877,0.633129
5,0.612213,0.931377,0.628209,0.630436


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=4375, training_loss=0.7563299464634486, metrics={'train_runtime': 309.4369, 'train_samples_per_second': 226.12, 'train_steps_per_second': 14.139, 'total_flos': 4602511459607040.0, 'train_loss': 0.7563299464634486, 'epoch': 5.0})

In [10]:
full_model.save_pretrained("./hugging_full_ft_model")
tokenizer.save_pretrained("./hugging_full_ft_model")
print("Full fine-tuned model saved.")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]

Full fine-tuned model saved.


---
## Part 2 — PEFT / LoRA Fine-Tuning

In [5]:
peft_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # sequence classification
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]  # XLM-R attention projection names
)

peft_model = get_peft_model(peft_base, lora_config)
peft_model.print_trainable_parameters()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6465.40it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,182,723 || all params: 279,228,678 || trainable%: 0.4236


In [6]:
peft_args = TrainingArguments(
    output_dir="./hf_peft_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4, # higher LR works better for LoRA
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    warmup_steps=200,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    bf16=True,
    logging_steps=50,
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

peft_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.871890,0.852120,0.596199,0.600797
2,0.843657,0.843025,0.614538,0.613505
3,0.807443,0.822265,0.621541,0.624587
4,0.777187,0.821186,0.631544,0.634178
5,0.773841,0.823712,0.623875,0.627499


TrainOutput(global_step=4375, training_loss=0.8320440194266183, metrics={'train_runtime': 212.6975, 'train_samples_per_second': 328.965, 'train_steps_per_second': 20.569, 'total_flos': 4666067398149120.0, 'train_loss': 0.8320440194266183, 'epoch': 5.0})

In [7]:
# Merge LoRA weights into base model and save as a standard model
merged_model = peft_model.merge_and_unload() # for Docker
merged_model.save_pretrained("./hugging_peft_model_merged")
tokenizer.save_pretrained("./hugging_peft_model_merged")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


('./hugging_peft_model_merged\\tokenizer_config.json',
 './hugging_peft_model_merged\\tokenizer.json')

---
## Part 3 — Evaluation on Validation Set

In [12]:
# Evaluate on validation set (test set is unlabelled)
def evaluate_on_val(trainer, model_label):
    predictions = trainer.predict(val_tok)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    print()
    print(f"{model_label} — Validation Set Results")
    print()
    print(classification_report(
        labels, preds,
        target_names=["Positive", "Negative", "Neutral"]
    ))
    return preds, labels

In [13]:
full_preds, true_labels = evaluate_on_val(full_trainer, "Full Fine-Tuning")
peft_preds, _ = evaluate_on_val(peft_trainer, "PEFT / LoRA")


Full Fine-Tuning — Validation Set Results

              precision    recall  f1-score   support

    Positive       0.68      0.72      0.70       982
    Negative       0.63      0.67      0.65       890
     Neutral       0.58      0.52      0.55      1127

    accuracy                           0.63      2999
   macro avg       0.63      0.64      0.63      2999
weighted avg       0.63      0.63      0.63      2999




PEFT / LoRA — Validation Set Results

              precision    recall  f1-score   support

    Positive       0.68      0.73      0.70       982
    Negative       0.63      0.67      0.65       890
     Neutral       0.58      0.52      0.55      1127

    accuracy                           0.63      2999
   macro avg       0.63      0.64      0.63      2999
weighted avg       0.63      0.63      0.63      2999



In [14]:
# Side-by-side comparison summary
full_f1  = f1_score(true_labels, full_preds, average="macro")
peft_f1  = f1_score(true_labels, peft_preds, average="macro")
full_acc = accuracy_score(true_labels, full_preds)
peft_acc = accuracy_score(true_labels, peft_preds)

print("\nComparison Summary (Validation Set)")
print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>10}")
print()
print(f"{'Full Fine-Tuning':<30} {full_acc:>10.4f} {full_f1:>10.4f}")
print(f"{'PEFT/LoRA':<30} {peft_acc:>10.4f} {peft_f1:>10.4f}")


Comparison Summary (Validation Set)
Model                            Accuracy   Macro F1

Full Fine-Tuning                   0.6289     0.6312
PEFT/LoRA                          0.6305     0.6327


---
## Part 4 — Inference on Unlabelled Test Set

In [ ]:
'''
# Generate predictions on the unlabelled test set - need to label the test set
def predict_on_test(trainer, model_label):
    predictions = trainer.predict(test_tok)
    preds = np.argmax(predictions.predictions, axis=-1)
    pred_labels = [ID2LABEL[p] for p in preds]

    print(f"\n> {model_label} — Test Set Prediction Distribution <")
    from collections import Counter
    print(Counter(pred_labels))
    return pred_labels

full_test_preds = predict_on_test(full_trainer, "Full Fine-Tuning")
peft_test_preds = predict_on_test(peft_trainer, "PEFT / LoRA")
'''

In [16]:
# Inference on sample Hinglish sentences
def predict_sentiment(text, model, tokenizer, id2label):
    device = next(model.parameters()).device
    inputs = tokenizer(
        text, return_tensors="pt",
        max_length=128, truncation=True, padding="max_length"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = torch.argmax(logits, dim=-1).item()
    confidence = torch.softmax(logits, dim=-1).max().item()
    return id2label[pred_id], round(confidence, 4)


sample_sentences = [
    "yaar aaj ka din bahut accha tha", # positive
    "mujhe bilkul pasand nahi yeh cheez", # negative
    "sab kuch theek hai, koi baat nahi", # neutral
]

print("\n> Full Fine-Tuning Inference <")
for s in sample_sentences:
    label, conf = predict_sentiment(s, full_model, tokenizer, ID2LABEL)
    print(f"  '{s}'\n  >> {label} (confidence: {conf})\n")

print("\n> PEFT/LoRA Inference <")
for s in sample_sentences:
    label, conf = predict_sentiment(s, peft_model, tokenizer, ID2LABEL)
    print(f"  '{s}'\n  >> {label} (confidence: {conf})\n")


> Full Fine-Tuning Inference <
  'yaar aaj ka din bahut accha tha'
  >> positive (confidence: 0.915)

  'mujhe bilkul pasand nahi yeh cheez'
  >> neutral (confidence: 0.9204)

  'sab kuch theek hai, koi baat nahi'
  >> neutral (confidence: 0.7676)


> PEFT/LoRA Inference <
  'yaar aaj ka din bahut accha tha'
  >> positive (confidence: 0.7451)

  'mujhe bilkul pasand nahi yeh cheez'
  >> neutral (confidence: 0.6737)

  'sab kuch theek hai, koi baat nahi'
  >> neutral (confidence: 0.7421)



Good average scores, but struggles with negative inputs!